# 5. Combining Profiles

## Purpose
Concatenate all per-well-FOV parquet files for a single patient into three
patient-level combined parquets (SC, organoid, nucleocentric).

This is **step 5 of Stage 4 (image-based profiling)**. It runs once per patient
and is typically submitted as a per-patient SLURM job.

## Inputs
- `data/{patient}/image_based_profiles/1.related_profiles/{well_fov}/`
  - `sc_profiles_{well_fov}_related.parquet`
  - `organoid_profiles_{well_fov}_related.parquet`
  - `nucleocentric_profiles_{well_fov}_related.parquet`

## Outputs
Three combined parquets in `data/{patient}/image_based_profiles/2.combined_profiles/`:

| File | Content |
|---|---|
| `sc.parquet` | All SC profiles stacked across FOVs |
| `organoid.parquet` | All organoid profiles stacked across FOVs |
| `nucleocentric.parquet` | All nucleocentric profiles stacked across FOVs |

## Notes
- Concatenation uses DuckDB `union_by_name=true`, which aligns columns by name
  rather than position. FOVs with missing columns (e.g. empty scaffold tables)
  will have those columns filled with NULL.
- Brightfield (BF) channel features are removed after concatenation as they are
  not part of the fluorescent cell painting panel and are not used in profiling.

In [1]:
import os
import pathlib

import duckdb
import pandas as pd
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0014_T1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
# set paths
profiles_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles"
).resolve(strict=True)
# output_paths
sc_merged_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/sc.parquet"
).resolve()
organoid_merged_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/organoid.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/nucleocentric.parquet"
).resolve()
organoid_merged_output_path.parent.mkdir(parents=True, exist_ok=True)

In [4]:
# Discover all per-FOV parquet files under 1.related_profiles/.
# The directory structure is 1.related_profiles/{well_fov}/*.parquet,
# so one wildcard level is sufficient.
profiles = list(profiles_path.rglob("*/*.parquet"))

In [5]:
# Split files by profile type using filename prefix.
# Expected prefixes: 'sc_', 'organoid_', 'nucleocentric_'.
sc_profiles = [str(x) for x in profiles if x.name.startswith("sc_")]
organoid_profiles = [str(x) for x in profiles if x.name.startswith("organoid_")]
nucleocentric_profiles = [
    str(x) for x in profiles if x.name.startswith("nucleocentric_")
]

In [6]:
for x in nucleocentric_profiles:
    df = pd.read_parquet(x)
    if df.isnull().any().any():
        print(f"Null values found in {x}")
df

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99,ParentOrganoid
0,257,C10-1,-0.314974,-0.316174,0.238469,-0.008768,-0.155256,0.190950,0.023767,-0.083596,...,0.883319,-5.715525,-2.876581,1.559442,0.312290,0.952051,3.002643,-4.167339,-1.677402,-1
1,514,C10-1,-0.488863,-0.252631,0.218513,-0.039571,-0.135434,0.030658,-0.090721,-0.010430,...,-0.892631,-6.424831,-3.211543,2.283450,-5.713564,0.695556,5.365023,-0.231492,-1.873856,1
2,771,C10-1,-0.467338,-0.239952,0.219353,-0.046014,-0.108846,0.041856,-0.076545,0.006468,...,0.098094,-6.672180,-2.968918,3.093965,-7.020364,1.473802,0.930933,-1.184521,-0.529366,1
3,1028,C10-1,-0.195418,-0.267018,0.126247,-0.043507,-0.114767,0.257694,0.003397,-0.145683,...,4.270934,-7.243169,2.599591,0.495510,-0.690967,-2.965625,6.857784,2.518691,-0.112033,-1
4,1285,C10-1,-0.428968,-0.108651,0.172778,-0.036185,0.006398,0.053850,-0.053953,-0.078261,...,3.148130,-6.224712,-2.419286,6.200053,0.547613,-0.412620,2.633893,-3.002506,1.181259,1
5,1542,C10-1,-0.081453,-0.269048,0.187046,-0.031576,-0.043143,0.241397,0.123766,-0.187622,...,4.838858,-9.338998,3.789076,8.516677,1.679702,3.476436,4.454591,-2.240638,-0.000733,1
6,1799,C10-1,-0.128135,0.014040,0.114629,-0.010768,0.020272,-0.036512,0.109075,-0.191876,...,1.297477,-12.410695,1.356477,8.786969,-2.507699,1.640216,0.661193,-2.984402,1.019717,1
7,2056,C10-1,-0.033046,-0.147866,0.127329,-0.200664,-0.124064,0.194111,0.075338,-0.208771,...,-1.365574,-8.453912,-3.141807,2.626937,-4.108184,-0.731863,2.228125,-0.379013,-0.652991,1
8,2313,C10-1,-0.094026,-0.180568,0.130076,-0.123790,-0.078242,0.272000,0.123896,-0.219325,...,4.402211,-10.991883,3.374835,6.054252,-1.054146,2.006402,1.156290,-1.037703,-0.136088,1
9,2570,C10-1,-0.114857,-0.203724,0.064930,0.034829,-0.128472,0.256379,-0.001898,-0.115626,...,5.504440,-5.511956,8.858730,2.981123,1.518283,0.252837,4.960571,0.586505,0.113625,1


In [7]:
# Concatenate per-FOV parquets for each profile type using DuckDB.
# union_by_name=true aligns columns by name rather than position, so FOVs with
# differing column sets (e.g. empty scaffold tables from notebook 1) are handled
# gracefully — missing columns are filled with NULL rather than causing an error.

with duckdb.connect() as conn:
    sc_profile = conn.execute(
        f"SELECT * FROM read_parquet({sc_profiles}, union_by_name=true)"
    ).df()
    organoid_profile = conn.execute(
        f"SELECT * FROM read_parquet({organoid_profiles}, union_by_name=true)"
    ).df()
    nucleocentric_profile = conn.execute(
        f"SELECT * FROM read_parquet({nucleocentric_profiles}, union_by_name=true)"
    ).df()

print(f"Single-cell profiles concatenated. Shape: {sc_profile.shape}")
print(f"Organoid profiles concatenated. Shape: {organoid_profile.shape}")
print(f"Nucleocentric profiles concatenated. Shape: {nucleocentric_profile.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Single-cell profiles concatenated. Shape: (2162, 11889)
Organoid profiles concatenated. Shape: (262, 3962)
Nucleocentric profiles concatenated. Shape: (2162, 3075)


## Remove all BF channels


In [8]:
# Remove brightfield (BF) channel features from all three profile types.
# BF is a transmitted-light channel not part of the fluorescent cell painting
# panel; its features are may not meaningful for morphological profiling.
# and are interpreted differently
# Note: if no BF columns exist in the data, these drops are no-ops.

bf_cols_sc = [col for col in sc_profile.columns if "BF" in col]
sc_profile = sc_profile.drop(columns=bf_cols_sc)
print(f"SC: dropped {len(bf_cols_sc)} BF columns. Shape: {sc_profile.shape}")

bf_cols_organoid = [col for col in organoid_profile.columns if "BF" in col]
organoid_profile = organoid_profile.drop(columns=bf_cols_organoid)
print(
    f"Organoid: dropped {len(bf_cols_organoid)} BF columns. Shape: {organoid_profile.shape}"
)

bf_cols_nucleocentric = [col for col in nucleocentric_profile.columns if "BF" in col]
nucleocentric_profile = nucleocentric_profile.drop(columns=bf_cols_nucleocentric)
print(
    f"Nucleocentric: dropped {len(bf_cols_nucleocentric)} BF columns. Shape: {nucleocentric_profile.shape}"
)

SC: dropped 0 BF columns. Shape: (2162, 11889)
Organoid: dropped 0 BF columns. Shape: (262, 3962)
Nucleocentric: dropped 0 BF columns. Shape: (2162, 3075)


In [9]:
sc_profile.to_parquet(sc_merged_output_path, index=False)
organoid_profile.to_parquet(organoid_merged_output_path, index=False)
nucleocentric_profile.to_parquet(nucleocentric_profile_output_path, index=False)

In [10]:
nucleocentric_profile

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99,ParentOrganoid
0,257,G8-1,-0.069569,-0.184811,0.048314,0.064896,-0.227413,0.250964,0.132686,-0.149804,...,4.397966,-8.548804,-0.670019,5.290964,-0.075234,-0.952333,7.367973,-1.459231,0.964480,1
1,514,G8-1,0.071577,-0.315487,0.126586,0.066663,-0.094124,0.310010,0.155788,-0.156147,...,4.146325,-6.469439,-1.077425,6.044081,-0.090697,-1.331949,6.247814,-0.649954,-0.220415,1
2,771,G8-1,-0.504519,-0.171609,0.201000,-0.067342,-0.029872,0.049545,-0.065619,-0.034082,...,3.994222,-9.295187,-2.610502,3.627539,0.032823,-3.106016,6.079479,-1.996334,2.673152,1
3,1028,G8-1,-0.374870,-0.111162,0.154739,0.005361,-0.062405,0.100905,0.035566,-0.165781,...,3.869776,-8.748277,-1.221812,5.858251,-2.792468,-0.767644,4.601731,-1.952397,0.503304,1
4,1285,G8-1,0.024705,-0.138621,0.122796,-0.005601,0.125955,0.027121,0.318451,-0.087292,...,3.911092,-4.565470,-3.044102,1.864853,-4.031749,0.816288,3.875162,-4.012982,-0.443435,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2157,2313,C10-1,-0.094026,-0.180568,0.130076,-0.123790,-0.078242,0.272000,0.123896,-0.219325,...,4.402211,-10.991883,3.374835,6.054252,-1.054146,2.006402,1.156290,-1.037703,-0.136088,1
2158,2570,C10-1,-0.114857,-0.203724,0.064930,0.034829,-0.128472,0.256379,-0.001898,-0.115626,...,5.504440,-5.511956,8.858730,2.981123,1.518283,0.252837,4.960571,0.586505,0.113625,1
2159,2827,C10-1,-0.021688,-0.106657,0.048318,-0.226410,0.112120,0.237900,0.034664,-0.082190,...,3.546839,-5.982894,-2.826436,3.022224,-7.846506,0.714891,0.354104,0.050066,-2.833983,1
2160,3084,C10-1,0.089536,-0.014970,0.107233,-0.218439,0.206013,0.200472,0.099758,-0.092002,...,1.656393,-8.709691,-1.942589,4.472564,-5.966566,-2.095769,3.379247,-1.611659,-0.867069,1
